In [46]:
import numpy as np 
import torch.nn.functional as F
import torch

Q = np.random.rand(10, 128)
K = np.random.rand(10, 128)
V = np.random.rand(10, 128)

def softmax(x):
    max_vals = np.max(x, axis=-1, keepdims=True)
    top = np.exp(x - max_vals) 
    bottom = np.sum(np.exp(x - max_vals), keepdims=True, axis=-1)
    return top / bottom

array([[0.33222212, 0.58847686, 0.53827998, ..., 0.61118302, 0.38393667,
        0.55864422],
       [0.31559087, 0.76056141, 0.49230375, ..., 0.54772511, 0.27354142,
        0.4355922 ],
       [0.27619014, 0.65004541, 0.51380143, ..., 0.72114757, 0.34894081,
        0.66670111],
       ...,
       [0.34004964, 0.68582139, 0.4981214 , ..., 0.53248195, 0.41072994,
        0.38821099],
       [0.31345927, 0.65560581, 0.46436217, ..., 0.61171303, 0.40081274,
        0.50752452],
       [0.31435787, 0.46320207, 0.54602829, ..., 0.68609189, 0.49812858,
        0.67987374]])

In [77]:
def softmax_causal(x):
    m_dim, n_dim = x.shape 
    assert len(x.shape) == 2 and "only work for dim2 for now"
    result = np.zeros(x.shape)

    for m in range(m_dim):
        max_val = np.max(x[m, 0:(m+1)])
        sum_val = np.sum(np.exp(x[m, 0:(m+1)] - np.array([max_val])))
        for n in range(n_dim):
            if n <= m: 
                result[m, n] = np.exp(x[m, n] - max_val) / sum_val 
            else: 
                result[m, n] = 0

    return result

def my_attention(Q, K, V): 
    dim_sequence, dim_model = Q.shape 
    _, dim_embedding = V.shape 
    output = np.zeros((dim_sequence, dim_embedding))
    
    for sequence in range(dim_sequence):
        for embedding in range(dim_embedding):
            for k in range(dim_sequence):
                output[sequence, embedding]+= softmax_causal(Q @ K.T)[sequence, k]  * V[k, embedding]
    return output

my_atten = my_attention(Q, K, V)
torch_atten = F.scaled_dot_product_attention(torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V), is_causal=True, scale=1)
print(f"My implementation:\n {my_atten}")
print(f"Pytorch implementation:\n {torch_atten}")
print(f"All close: {torch.allclose(torch.from_numpy(my_atten), torch_atten)}")

My implementation:
 [[0.13953109 0.64543115 0.78326762 ... 0.81252684 0.20055324 0.59576417]
 [0.47404972 0.50429656 0.62361089 ... 0.13456489 0.34564128 0.15737517]
 [0.37886008 0.49784665 0.61869661 ... 0.2862572  0.29065726 0.24720975]
 ...
 [0.37366014 0.40055783 0.53569907 ... 0.60352697 0.78279581 0.45307162]
 [0.27738256 0.54641288 0.46134324 ... 0.76736861 0.49327973 0.70198294]
 [0.31435787 0.46320207 0.54602829 ... 0.68609189 0.49812858 0.67987374]]
Pytorch implementation:
 tensor([[0.1395, 0.6454, 0.7833,  ..., 0.8125, 0.2006, 0.5958],
        [0.4740, 0.5043, 0.6236,  ..., 0.1346, 0.3456, 0.1574],
        [0.3789, 0.4978, 0.6187,  ..., 0.2863, 0.2907, 0.2472],
        ...,
        [0.3737, 0.4006, 0.5357,  ..., 0.6035, 0.7828, 0.4531],
        [0.2774, 0.5464, 0.4613,  ..., 0.7674, 0.4933, 0.7020],
        [0.3144, 0.4632, 0.5460,  ..., 0.6861, 0.4981, 0.6799]],
       dtype=torch.float64)
All close: True


In [76]:
my_atten.shape

(10, 128)